# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SIDRAATTIQUE/flyrank-machine-learning/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os

HF_TOKEN = userdata.get('HF_Token')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("✅ Connected")

✅ Connected


In [3]:
from huggingface_hub import snapshot_download

print("Downloading Feb-March 2026 partitions locally...")
local_path = snapshot_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    token=HF_TOKEN,
    allow_patterns=[
        'fact_content_daily_performance/month=2026-02/*.parquet',
        'fact_content_daily_performance/month=2026-03/*.parquet',
    ]
)
print(f"✅ Downloaded to: {local_path}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [4]:
LOCAL_FACT = f"read_parquet('{local_path}/fact_content_daily_performance/month=2026-0*/*.parquet')"

print("Building feature frame...")

feature_frame = con.sql(f"""
    WITH features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_impressions ELSE 0 END)         AS imp_last30,
            SUM(CASE WHEN report_date < '2026-03-01'
                THEN gsc_impressions ELSE 0 END)          AS imp_prev30,
            SUM(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_clicks ELSE 0 END)               AS clk_last30,
            SUM(CASE WHEN report_date < '2026-03-01'
                THEN gsc_clicks ELSE 0 END)               AS clk_prev30,
            AVG(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_avg_position END)                AS pos_last30,
            AVG(CASE WHEN report_date < '2026-03-01'
                THEN gsc_avg_position END)                AS pos_prev30,
            STDDEV(gsc_avg_position)                      AS pos_volatility,
            COUNT(DISTINCT report_date)                   AS days_of_data
        FROM {LOCAL_FACT}
        WHERE report_date >= '2026-02-01'
          AND report_date <= '2026-03-31'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 10
    )
    SELECT * FROM features
""").df()

# Engineered features
feature_frame['ctr_last30'] = (
    feature_frame['clk_last30'] / feature_frame['imp_last30'].replace(0, np.nan)
)
feature_frame['ctr_prev30'] = (
    feature_frame['clk_prev30'] / feature_frame['imp_prev30'].replace(0, np.nan)
)
feature_frame['ctr_change'] = feature_frame['ctr_last30'] - feature_frame['ctr_prev30']
feature_frame['pos_change'] = feature_frame['pos_last30'] - feature_frame['pos_prev30']

# Label
feature_frame['is_declining'] = (
    feature_frame['imp_last30'] < 0.8 * feature_frame['imp_prev30']
).astype(int)

print(f"✅ Rows: {len(feature_frame):,}")
print(f"✅ Unique clients: {feature_frame['client_hash_id'].nunique()}")
print(f"✅ Declining pages: {feature_frame['is_declining'].sum():,}")
display(feature_frame.head())

Building feature frame...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Rows: 119,340
✅ Unique clients: 42
✅ Declining pages: 30,039


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,clk_prev30,pos_last30,pos_prev30,pos_volatility,days_of_data,ctr_last30,ctr_prev30,ctr_change,pos_change,is_declining
0,client_3ffa76342f366962,content_78fce5fb92fdf949,29.0,15.0,0.0,0.0,4.583333,4.111111,2.729518,49,0.0,0.000000,0.000000,0.472222,0
1,client_3ffa76342f366962,content_cae1d5374958a649,43.0,96.0,0.0,0.0,7.352381,5.262333,2.871747,58,0.0,0.000000,0.000000,2.090047,1
2,client_3ffa76342f366962,content_dd66eecf9626cab8,255.0,235.0,0.0,0.0,5.853611,6.407819,1.834977,59,0.0,0.000000,0.000000,-0.554208,0
3,client_3ffa76342f366962,content_c51f1e8ef5502177,15.0,18.0,0.0,0.0,7.766667,6.961538,2.712813,47,0.0,0.000000,0.000000,0.805128,0
4,client_3ffa76342f366962,content_0674cc4ae0f68a90,11.0,74.0,0.0,1.0,6.333333,8.063910,2.505196,54,0.0,0.013514,-0.013514,-1.730576,1


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# ============================================
# IMPROVED MODEL: Balanced Random Forest
# class_weight='balanced' fixes the imbalance problem
# ============================================
model_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight='balanced',  # This is the key fix
    random_state=42,
    n_jobs=-1
)
model_balanced.fit(X_train, y_train)
model_preds_balanced = model_balanced.predict(X_test)

model_accuracy  = accuracy_score(y_test, model_preds_balanced)
model_precision = precision_score(y_test, model_preds_balanced, zero_division=0)
model_recall    = recall_score(y_test, model_preds_balanced, zero_division=0)
model_f1        = f1_score(y_test, model_preds_balanced, zero_division=0)

# Baseline numbers (same as before)
test_rows = model_data.iloc[test_idx]
baseline_preds = (
    (test_rows['imp_last30'] < 0.8 * test_rows['imp_prev30']) &
    (test_rows['pos_volatility'] < 2.0)
).astype(int).values

baseline_accuracy  = accuracy_score(y_test, baseline_preds)
baseline_precision = precision_score(y_test, baseline_preds, zero_division=0)
baseline_recall    = recall_score(y_test, baseline_preds, zero_division=0)
baseline_f1        = f1_score(y_test, baseline_preds, zero_division=0)

# Dumb baseline
dumb_preds    = np.full(len(y_test), y_test.mode()[0])
dumb_accuracy = accuracy_score(y_test, dumb_preds)

# Updated comparison table
comparison = pd.DataFrame({
    'Model': [
        'Dumb Baseline (always majority)',
        'Week-4 Rule Baseline',
        'Random Forest Unbalanced',
        'Random Forest Balanced (Week-5)'
    ],
    'Accuracy':  [dumb_accuracy,  baseline_accuracy,  0.743, model_accuracy],
    'Precision': [np.nan,         baseline_precision, 0.677, model_precision],
    'Recall':    [np.nan,         baseline_recall,    0.002, model_recall],
    'F1':        [np.nan,         baseline_f1,        0.004, model_f1],
})

print("UPDATED MODEL vs BASELINE COMPARISON:")
display(comparison)

print("\nFull Balanced Random Forest Report:")
print(classification_report(y_test, model_preds_balanced, digits=3))

UPDATED MODEL vs BASELINE COMPARISON:


,Model,Accuracy,Precision,Recall,F1
0,Dumb Baseline (always majority),0.743149,NaN,NaN,NaN
1,Week-4 Rule Baseline,0.764716,1.000000,0.083965,0.154921
2,Random Forest Unbalanced,0.743000,0.677000,0.002000,0.004000
3,Random Forest Balanced (Week-5),0.561630,0.317681,0.615705,0.419115



Full Balanced Random Forest Report:
              precision    recall  f1-score   support

           0      0.803     0.543     0.648     27119
           1      0.318     0.616     0.419      9373

    accuracy                          0.562     36492
   macro avg      0.561     0.579     0.534     36492
weighted avg      0.679     0.562     0.589     36492



## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

**Method chosen: Random Forest Classifier with class_weight='balanced'**

I initially trained an unbalanced Random Forest, which revealed a
critical problem: my dataset has 74% stable pages vs 26% declining
pages. The unbalanced model achieved 74.3% accuracy simply by
predicting "stable" for almost everything — a recall of 0.002
means it found only 21 of 9,373 truly declining pages. This is
a useless model for a content team trying to find refresh opportunities.

I switched to a balanced Random Forest (class_weight='balanced'),
which internally reweights training examples so the model pays equal
attention to both classes. This is the right choice because:

1. **Missing a declining page is the costlier error** in content
   refresh — an editor wants to FIND opportunities, not just confirm
   the obvious ones. Recall matters more than raw accuracy here.

2. **Non-linear interactions:** Features like `pos_change`, `ctr_change`,
   and `pos_volatility` interact in ways Logistic Regression cannot
   capture. The permutation importance confirmed `pos_change` is the
   strongest signal — a directional feature my Week-4 rule didn't
   even use.

3. **Fair complexity:** I chose Random Forest over Gradient Boosting
   because with only 37 clients in a grouped split, simpler is
   more honest — "don't reward complexity alone."

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [5]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    'imp_prev30', 'pos_volatility', 'ctr_change', 'pos_change', 'days_of_data'
]

model_data = feature_frame.dropna(subset=feature_cols + ['is_declining']).copy()

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

print(f"Total clients in dataset: {groups.nunique()}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test  = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

print(f"\nTrain: {len(X_train):,} rows | {model_data.iloc[train_idx]['client_hash_id'].nunique()} clients")
print(f"Test:  {len(X_test):,} rows  | {model_data.iloc[test_idx]['client_hash_id'].nunique()} clients")
print(f"\nTest label balance:")
print(y_test.value_counts(normalize=True))

Total clients in dataset: 37

Train: 75,002 rows | 27 clients
Test:  36,492 rows  | 10 clients

Test label balance:
is_declining
0    0.743149
1    0.256851
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_score, recall_score
)

# ============================================
# MODEL: Random Forest (Week 5)
# ============================================
model = RandomForestClassifier(
    n_estimators=200, max_depth=6,
    random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

model_accuracy  = accuracy_score(y_test, model_preds)
model_precision = precision_score(y_test, model_preds, zero_division=0)
model_recall    = recall_score(y_test, model_preds, zero_division=0)

# ============================================
# BASELINE: Week-4 Rule (same logic, no peeking at y!)
# Rule: decline > 20% AND stable position (volatility < 2.0)
# ============================================
test_rows = model_data.iloc[test_idx]
baseline_preds = (
    (test_rows['imp_last30'] < 0.8 * test_rows['imp_prev30']) &
    (test_rows['pos_volatility'] < 2.0)
).astype(int).values

baseline_accuracy  = accuracy_score(y_test, baseline_preds)
baseline_precision = precision_score(y_test, baseline_preds, zero_division=0)
baseline_recall    = recall_score(y_test, baseline_preds, zero_division=0)

# ============================================
# DUMB BASELINE: Always predict majority class
# ============================================
majority_class = y_test.mode()[0]
dumb_preds     = np.full(len(y_test), majority_class)
dumb_accuracy  = accuracy_score(y_test, dumb_preds)

# ============================================
# COMPARISON TABLE
# ============================================
comparison = pd.DataFrame({
    'Model': [
        'Dumb Baseline (always majority)',
        'Week-4 Rule Baseline',
        'Random Forest (Week-5)'
    ],
    'Accuracy':  [dumb_accuracy,  baseline_accuracy,  model_accuracy],
    'Precision': [np.nan,         baseline_precision, model_precision],
    'Recall':    [np.nan,         baseline_recall,    model_recall],
})

print("MODEL vs BASELINE COMPARISON (same grouped test set):")
display(comparison)

print("\nFull Random Forest Report:")
print(classification_report(y_test, model_preds, digits=3))

MODEL vs BASELINE COMPARISON (same grouped test set):


,Model,Accuracy,Precision,Recall
0,Dumb Baseline (always majority),0.743149,NaN,NaN
1,Week-4 Rule Baseline,0.764716,1.000000,0.083965
2,Random Forest (Week-5),0.743451,0.677419,0.002240



Full Random Forest Report:
              precision    recall  f1-score   support

           0      0.744     1.000     0.853     27119
           1      0.677     0.002     0.004      9373

    accuracy                          0.743     36492
   macro avg      0.710     0.501     0.429     36492
weighted avg      0.727     0.743     0.635     36492



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
from sklearn.inspection import permutation_importance

# Feature importance
perm = permutation_importance(
    model, X_test, y_test,
    n_repeats=10, random_state=42
)
importance_df = pd.DataFrame({
    'feature':    feature_cols,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print("Permutation Feature Importance:")
display(importance_df)

# Error analysis
X_test_copy = X_test.copy()
X_test_copy['actual']    = y_test.values
X_test_copy['predicted'] = model_preds
errors = X_test_copy[X_test_copy['actual'] != X_test_copy['predicted']]

print(f"\nTotal errors: {len(errors):,} / {len(X_test):,} ({len(errors)/len(X_test)*100:.1f}%)")
print(f"False Positives (flagged but stable): {((X_test_copy['predicted']==1) & (X_test_copy['actual']==0)).sum()}")
print(f"False Negatives (missed but declining): {((X_test_copy['predicted']==0) & (X_test_copy['actual']==1)).sum()}")
print("\nSample misclassified rows:")
display(errors.head(10))

Permutation Feature Importance:


,feature,importance
3,pos_change,0.000902
2,ctr_change,0.000260
1,pos_volatility,0.000132
0,imp_prev30,0.000041
4,days_of_data,-0.000033



Total errors: 9,362 / 36,492 (25.7%)
False Positives (flagged but stable): 10
False Negatives (missed but declining): 9352

Sample misclassified rows:


,imp_prev30,pos_volatility,ctr_change,pos_change,days_of_data,actual,predicted
1,96.0,2.871747,0.000000,2.090047,58,1,0
4,74.0,2.505196,-0.013514,-1.730576,54,1,0
5,30.0,3.228298,0.042857,2.954209,53,1,0
7,12.0,3.577440,0.142857,3.387500,47,1,0
8,100.0,2.014463,0.041224,-0.067333,57,1,0
11,198.0,1.891913,0.036364,-0.101940,57,1,0
13,10.0,12.502949,0.250000,-8.023810,43,1,0
14,12.0,13.592007,0.000000,15.152778,49,1,0
15,49.0,2.464151,0.000000,0.044949,53,1,0
16,10.0,2.941954,0.000000,-4.722222,39,1,0


## 4. Errors and interpretation

### What the model leans on

The permutation importance table shows `pos_change` is the strongest
signal, followed by `ctr_change` and `pos_volatility`. This is a
meaningful finding: the model learned that CHANGE in position
(not just its absolute value) is the best predictor of whether
a page is declining. My Week-4 baseline only used the absolute
level of volatility — the model found a better signal by looking
at direction of change.

### Where the model is wrong

The balanced model made 9,362 errors out of 36,492 test rows (25.7%):

- **False Positives (flagged but stable): 10 only**
  Very few false alarms — when the model says "refresh this," it
  is usually right.

- **False Negatives (missed declining pages): 9,352**
  The model misses many declining pages. Looking at the misclassified
  rows, most have high `pos_volatility` (2.5 to 13.5) — pages with
  very unstable rankings are hard to classify because their decline
  may be temporary noise rather than a real content problem.

### The precision-recall tradeoff

This is the key finding of this modeling week:

| Model | Precision | Recall | F1 | Interpretation |
|---|---|---|---|---|
| Week-4 Rule | 1.000 | 0.084 | 0.155 | Perfect but finds almost nothing |
| RF Balanced | 0.318 | 0.616 | 0.419 | Finds more but flags more noise |

**The F1 score tells the honest story:** The balanced Random Forest
(F1 = 0.419) is 2.7x better than the Week-4 rule (F1 = 0.155) at
the combined task of finding AND correctly flagging declining pages.

However, precision = 0.318 means roughly 2 out of every 3 pages the
model flags for refresh are actually stable. For a content team with
limited editor time, this false positive rate may be too costly.

### Does complexity help?

Yes — but with a caveat. The balanced Random Forest meaningfully
outperforms the Week-4 rule on F1 (0.419 vs 0.155), proving the
model learned patterns beyond the simple decay+volatility rule.
However, the unbalanced RF (0.004 F1) shows that model complexity
alone is worthless without proper handling of class imbalance.

The right production decision depends on the editor's tolerance
for false positives:
- If editors want a SHORT list they can fully trust → use Week-4 rule
- If editors want to FIND MORE opportunities and can filter noise →
  use balanced RF

### Careful language

These results are observed on 119,340 content items from 37 clients
across one two-month window (Feb–March 2026) using a grouped-by-client
split (27 clients train, 10 clients test). The grouped split confirms
the model generalizes across clients, not just within them. All findings
are directional and decision-support only — not proof of causal impact
or guaranteed future performance on unseen time windows.

In [9]:
# ============================================
# SECTION 4: Errors and Interpretation
# Using the BALANCED model (model_balanced)
# ============================================

from sklearn.inspection import permutation_importance

# Permutation importance on balanced model
perm = permutation_importance(
    model_balanced, X_test, y_test,
    n_repeats=10, random_state=42
)

importance_df = pd.DataFrame({
    'feature':    feature_cols,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print("Permutation Importance (Balanced Model):")
display(importance_df)

# Error breakdown
X_test_copy = X_test.copy()
X_test_copy['actual']    = y_test.values
X_test_copy['predicted'] = model_preds_balanced

errors = X_test_copy[X_test_copy['actual'] != X_test_copy['predicted']]

fp = ((X_test_copy['predicted']==1) & (X_test_copy['actual']==0)).sum()
fn = ((X_test_copy['predicted']==0) & (X_test_copy['actual']==1)).sum()

print(f"\nTotal errors:                          {len(errors):,} / {len(X_test):,} ({len(errors)/len(X_test)*100:.1f}%)")
print(f"False Positives (flagged but stable):  {fp:,}")
print(f"False Negatives (missed declining):    {fn:,}")
print("\nSample misclassified rows:")
display(errors.head(10))

Permutation Importance (Balanced Model):


,feature,importance
2,ctr_change,0.068042
1,pos_volatility,0.049589
3,pos_change,0.006552
0,imp_prev30,-0.023975
4,days_of_data,-0.024912



Total errors:                          15,997 / 36,492 (43.8%)
False Positives (flagged but stable):  12,395
False Negatives (missed declining):    3,602

Sample misclassified rows:


,imp_prev30,pos_volatility,ctr_change,pos_change,days_of_data,actual,predicted
0,15.0,2.729518,0.000000,0.472222,49,0,1
2,235.0,1.834977,0.000000,-0.554208,59,0,1
3,18.0,2.712813,0.000000,0.805128,47,0,1
4,74.0,2.505196,-0.013514,-1.730576,54,1,0
5,30.0,3.228298,0.042857,2.954209,53,1,0
6,26.0,11.904368,-0.021512,-1.069052,50,0,1
7,12.0,3.577440,0.142857,3.387500,47,1,0
10,11.0,7.336671,0.000000,7.801587,46,0,1
13,10.0,12.502949,0.250000,-8.023810,43,1,0
16,10.0,2.941954,0.000000,-4.722222,39,1,0


## 4. Errors and interpretation

### What the model leans on

The permutation importance table (balanced model) shows:

1. ctr_change     → 0.068 (strongest signal by far)
2. pos_volatility → 0.050 (second strongest)
3. pos_change     → 0.007 (weak but positive)
4. imp_prev30     → -0.024 (hurts the model — removing it would help)
5. days_of_data   → -0.025 (hurts the model — removing it would help)

The most important finding: ctr_change (how CTR shifted
month-over-month) is the #1 signal — NOT raw impression volume.
This is a meaningful discovery: a page losing clicks relative to
its visibility is a stronger refresh signal than a page that simply
had fewer impressions. My Week-4 baseline did not use ctr_change
at all — this is the key thing the model learned that the rule
could not see.

The two negative-importance features (imp_prev30 and days_of_data)
are actually hurting the model. This means they add noise rather
than signal — removing them would likely improve performance.

### Where the model is wrong

Total errors: 15,997 out of 36,492 test rows (43.8%)

- False Positives (flagged but actually stable): 12,395
  The model is too aggressive — it flags many stable pages
  for refresh. Looking at the misclassified rows, false positives
  tend to have high pos_volatility (2.7 to 11.9) and zero
  ctr_change. The model interprets position instability as a
  decline signal even when CTR hasn't actually changed.

- False Negatives (missed declining pages): 3,602
  The model misses 3,602 truly declining pages. These tend to
  have moderate volatility and small negative pos_change —
  subtle declines the model isn't confident enough to flag.

### The precision-recall tradeoff (the honest story)

| Model            | Precision | Recall | F1    | Interpretation              |
|------------------|-----------|--------|-------|-----------------------------|
| Week-4 Rule      | 1.000     | 0.084  | 0.155 | Perfect but finds almost nothing |
| RF Unbalanced    | 0.677     | 0.002  | 0.004 | Useless — predicted stable always |
| RF Balanced      | 0.318     | 0.616  | 0.419 | Finds more, but too many false alarms |

The F1 score is the honest comparison metric here because accuracy
is misleading on an imbalanced dataset (74% stable vs 26% declining).
The balanced Random Forest (F1 = 0.419) is 2.7x better than the
Week-4 rule (F1 = 0.155) at the combined task of finding AND
correctly flagging declining pages.

However, precision = 0.318 means roughly 2 out of every 3 pages
the model flags are actually stable. For a content team with
limited editor time, this false positive rate is high.

### What I would do next to improve this

The negative permutation importance of imp_prev30 and days_of_data
suggests a clear next step: remove these features and retrain.
The model is currently confused by volume and history-length signals
that add noise rather than predictive power.

A stronger model would:
1. Drop imp_prev30 and days_of_data from feature_cols
2. Add a query-level signal from fact_content_query_90d
   (how many queries drive this page's traffic)
3. Test a threshold on the model's predict_proba output
   instead of a hard 0/1 decision — this lets the content
   team control the precision/recall tradeoff directly

### Does complexity help?

Yes — but conditionally. The balanced RF (F1 = 0.419) genuinely
outperforms the Week-4 rule (F1 = 0.155), proving the model learned
real patterns — specifically that ctr_change is a stronger signal
than the simple decay+volatility rule I wrote by hand.

However, the model's high false positive rate (12,395 false alarms)
means it is not yet production-ready as a standalone tool. The
right production decision depends on the editor's tolerance:
- Short trusted list → use Week-4 rule (precision = 1.0)
- Find more opportunities, accept some noise → use balanced RF

### Careful language

These results are observed on 119,340 content items from 37 clients
across one two-month window (Feb–March 2026) using a grouped-by-client
split (27 clients train, 10 clients test). The grouped split confirms
the model generalizes across unseen clients. All findings are
directional and decision-support only — not proof of causal impact
or guaranteed future performance on unseen time windows or
different client portfolios.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.